# Inspect Run9, Run10, Run11

This notebook analyzes the scheduler/model-size experiments and compares them against the clean U64 baseline `run16`.

| Run | Config | Model | Change | Epochs |
|---|---|---|---|---:|
| `run16` | `config_run16_u64_baseline.yaml` | U64 | clean baseline rerun | 120 |
| `run9` | `config_run9.yaml` | U64 | StepLR scheduler | 120 |
| `run10` | `config_run10_production.yaml` | U128 | deeper/wider production model | 500 |
| `run11` | `config_run11_unet256_production.yaml` | U256-width | largest UNet width | 500 |

Important: `run11` is not 256 x 256 images. The images are still `sample_size=128`. Here, “U256” means the first UNet channel width is 256: `[256, 512, 512, 512]`.


## What To Run On A Cluster

Submit the matching Slurm scripts, not the YAML directly:

```bash
cd /path/to/diffusion-models-simulation-data
sbatch batch_scripts/train_diffusion_run9.sbatch
sbatch batch_scripts/train_diffusion_run10_production.sbatch
sbatch batch_scripts/train_diffusion_run11_unet256_production.sbatch
```

If GPU slots are limited, do not submit all at once. `run11` is the most expensive and most likely to OOM.


In [ ]:
from pathlib import Path
import sys
import copy
import json

import numpy as np
import torch
import yaml
import matplotlib.pyplot as plt

PROJECT_DIR = Path.cwd()

COSMO_DIFFUSION_DIR = PROJECT_DIR / "cosmo_diffusion"
if str(COSMO_DIFFUSION_DIR) not in sys.path:
    sys.path.insert(0, str(COSMO_DIFFUSION_DIR))

from cosmodiff import utils
from cosmodiff.optim import generate, build_pca_encoder, compute_fid, compute_kid

print("cosmodiff utils:", utils.__file__)
if "centered_maxabs" not in open(utils.__file__).read():
    raise RuntimeError("This kernel is using an old utils.py. Sync the patched file to the cluster, then restart the kernel.")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PROJECT_DIR:", PROJECT_DIR)
print("DEVICE:", DEVICE)
if DEVICE.type != "cuda":
    print("WARNING: use a GPU kernel on the cluster before loading CUDA checkpoints.")


In [ ]:
RUNS = {
    "run16": {
        "config": PROJECT_DIR / "configs" / "templates" / "u64_lh_template.yaml",
        "label": "run16: U64 baseline",
        "color": "tab:green",
        "style": "-",
    },
    "run9": {
        "config": PROJECT_DIR / "configs" / "templates" / "u64_lh_template.yaml",
        "label": "run9: U64 StepLR",
        "color": "tab:blue",
        "style": "-",
    },
    "run10": {
        "config": PROJECT_DIR / "configs" / "templates" / "u128_lh_template.yaml",
        "label": "run10: U128 production",
        "color": "tab:red",
        "style": "-",
    },
    "run11": {
        "config": PROJECT_DIR / "configs" / "templates" / "u256_lh_template.yaml",
        "label": "run11: U256-width production",
        "color": "tab:purple",
        "style": "-",
    },
}

N_GEN = 20
N_REAL = 128
NBINS = 25
SEED = 123

for run_name, meta in RUNS.items():
    print(run_name, "config exists:", meta["config"].exists(), meta["config"])


## Config Summary

Use this table first. It tells you what actually changed. In particular, `run9` is not the U256 run; it is U64 with a different LR scheduler.


In [ ]:
def load_config(path):
    with open(path) as f:
        return yaml.safe_load(f)

rows = []
for run_name, meta in RUNS.items():
    cfg = load_config(meta["config"])
    data = cfg["data"]
    model = cfg["model"]["kwargs"]
    opt = cfg["optimizer"]["kwargs"]
    train = cfg["train"]
    sched = cfg.get("lr_scheduler", {})
    rows.append({
        "run": run_name,
        "sample_size": model.get("sample_size"),
        "channels": model.get("block_out_channels"),
        "layers_per_block": model.get("layers_per_block"),
        "epochs": train.get("num_epochs"),
        "checkpoint_every": train.get("checkpoint_every_n_epochs"),
        "batch_size": train.get("batch_size"),
        "grad_accum": train.get("gradient_accumulation_steps"),
        "effective_batch": train.get("batch_size") * train.get("gradient_accumulation_steps"),
        "lr": opt.get("lr"),
        "scheduler": sched.get("class"),
        "scheduler_kwargs": sched.get("kwargs"),
        "zthin": data.get("zthin"),
        "normalization": data.get("normalization"),
        "center": (data.get("norm_kwargs") or {}).get("center"),
        "augmentations": list((cfg.get("augmentations") or {}).keys()),
        "output_dir": cfg["io"]["output_dir"],
    })

try:
    import pandas as pd
    display(pd.DataFrame(rows))
except Exception:
    for row in rows:
        print(json.dumps(row, indent=2))


## Check What Finished

This cell checks the latest checkpoint for each run. The notebook skips missing checkpoints, so you can run it while jobs are still queued/running.


In [ ]:
def checkpoint_epoch(path):
    return int(Path(path).name.split("-")[-1])


def latest_checkpoint_from_config(cfg):
    output_dir = Path(cfg["io"]["output_dir"])
    ckpt = utils.find_latest_checkpoint(str(output_dir))
    return output_dir, ckpt

status_rows = []
for run_name, meta in RUNS.items():
    cfg = load_config(meta["config"])
    output_dir, ckpt = latest_checkpoint_from_config(cfg)
    target_last_epoch = cfg["train"]["num_epochs"] - 1
    if ckpt is None:
        loaded_epoch = None
        complete = False
    else:
        loaded_epoch = checkpoint_epoch(ckpt)
        complete = loaded_epoch >= target_last_epoch
    status_rows.append({
        "run": run_name,
        "output_dir": str(output_dir),
        "latest_checkpoint": None if ckpt is None else Path(ckpt).name,
        "latest_epoch": loaded_epoch,
        "target_epoch": target_last_epoch,
        "complete": complete,
    })

try:
    import pandas as pd
    display(pd.DataFrame(status_rows))
except Exception:
    for row in status_rows:
        print(json.dumps(row, indent=2))


## Load Checkpoints

If a production run has only an early checkpoint, the samples are undertrained diagnostics, not final results.


In [ ]:
def latest_metrics_path(output_dir, checkpoint_dir=None):
    candidates = []
    if checkpoint_dir is not None:
        ckpt_metrics = Path(checkpoint_dir) / "metrics.json"
        if ckpt_metrics.exists():
            candidates.append(ckpt_metrics)
    candidates.extend(Path(output_dir).glob("metrics_epoch_*.json"))
    if not candidates:
        return None
    return sorted(candidates, key=lambda p: p.stat().st_mtime)[-1]

run_state = {}
for run_name, meta in RUNS.items():
    cfg = load_config(meta["config"])
    output_dir = Path(cfg["io"]["output_dir"])
    ckpt = utils.find_latest_checkpoint(str(output_dir))

    print(f"\n{run_name}")
    print("  output_dir:", output_dir)
    print("  latest checkpoint:", ckpt)
    if ckpt is None:
        print("  SKIP: no checkpoint yet. Train this run first.")
        continue

    model, scheduler, optimizer, lr_scheduler, augmentations = utils.load_checkpoint(ckpt)
    model.to(DEVICE)
    model.eval()

    metrics_path = latest_metrics_path(output_dir, ckpt)
    metrics = utils.read_metrics(str(metrics_path)) if metrics_path is not None else None
    target_last_epoch = cfg["train"]["num_epochs"] - 1
    loaded_epoch = checkpoint_epoch(ckpt)

    print("  loaded epoch:", loaded_epoch)
    print("  target final epoch:", target_last_epoch)
    if loaded_epoch < target_last_epoch:
        print("  WARNING: checkpoint is not final yet; samples may be undertrained.")
    print("  metrics:", metrics_path)
    print("  augmentation restored:", augmentations)

    run_state[run_name] = {
        "config": cfg,
        "output_dir": output_dir,
        "checkpoint": Path(ckpt),
        "model": model,
        "scheduler": scheduler,
        "metrics": metrics,
        "metrics_path": metrics_path,
        "loaded_epoch": loaded_epoch,
    }

print("\nloaded runs:", list(run_state))


## Training Curves

For production runs, do not compare final sample quality until the epoch count is comparable. First check whether the loss is still rapidly decreasing.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for run_name, state in run_state.items():
    metrics = state["metrics"]
    if metrics is None:
        continue
    color = RUNS[run_name]["color"]
    label = RUNS[run_name]["label"]
    style = RUNS[run_name].get("style", "-")

    if "epoch_loss" in metrics:
        axes[0].plot(metrics["epoch_loss"], label=label, color=color, ls=style, lw=2)
    if "epoch_lr" in metrics:
        axes[1].plot(metrics["epoch_lr"], label=label, color=color, ls=style, lw=2)

axes[0].set_title("Epoch loss")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("noise-prediction MSE")
axes[0].legend()

axes[1].set_title("Learning rate")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("lr")
axes[1].legend()
fig.tight_layout()
plt.show()


## Load Real Reference Data

Each run's real data are loaded using that run's normalization. These runs all use mean-centered maxabs and `zthin=4`, so the reference should be nearly identical.


In [ ]:
real_images = {}
for run_name, state in run_state.items():
    data_config = copy.deepcopy(state["config"])
    data_config.setdefault("global", {})["device"] = "cpu"
    data_config.setdefault("data", {})["keep_on_cpu"] = True

    dataset = utils.parse_config_data(data_config)
    imgs = dataset.arrays.detach().cpu()

    if len(imgs) > N_REAL:
        idx = torch.linspace(0, len(imgs) - 1, N_REAL).long()
        imgs = imgs[idx]

    real_images[run_name] = imgs
    print(f"{run_name}: real_images shape = {tuple(imgs.shape)}")
    print(f"  range = [{imgs.min().item():.4f}, {imgs.max().item():.4f}]")
    print(f"  mean/std = {imgs.mean().item():.6f} / {imgs.std().item():.6f}")
    print(f"  frac(|x| >= 0.999) = {((imgs.abs() >= 0.999).float().mean().item()):.8e}")


## Generate Samples

This uses each checkpoint's saved DDPM scheduler. Use the same seed across runs so visual differences are easier to compare.


In [ ]:
generated = {}
for run_name, state in run_state.items():
    model = state["model"]
    scheduler = state["scheduler"]

    g = torch.Generator(device=DEVICE).manual_seed(SEED)
    with torch.no_grad():
        samples = generate(
            model,
            scheduler,
            batch_size=N_GEN,
            image_shape=(1, 128, 128),
            device=DEVICE,
            generator=g,
        ).detach().cpu()

    generated[run_name] = samples
    print(f"\n{run_name}")
    print(f"  generated shape = {tuple(samples.shape)}")
    print(f"  range = [{samples.min().item():.4f}, {samples.max().item():.4f}]")
    print(f"  mean/std = {samples.mean().item():.6f} / {samples.std().item():.6f}")
    print(f"  frac(|x| >= 0.999) = {((samples.abs() >= 0.999).float().mean().item()):.8e}")


## Visual Samples

The display range is chosen from real+generated percentiles for each run. This avoids misleading dim images from a fixed `[-1, 1]` display scale.


In [ ]:
def run_vlim(*tensors, q_low=0.5, q_high=99.5):
    vals = torch.cat([t.detach().cpu().flatten() for t in tensors])
    return torch.quantile(vals, torch.tensor([q_low / 100, q_high / 100])).tolist()


def plot_real_fake_grid(run_name, n=8):
    real = real_images[run_name][:n]
    fake = generated[run_name][:n]
    vmin, vmax = run_vlim(real, fake)

    fig, axes = plt.subplots(2, n, figsize=(1.8 * n, 3.8))
    for i in range(n):
        axes[0, i].imshow(real[i, 0], cmap="magma", vmin=vmin, vmax=vmax)
        axes[0, i].axis("off")
        axes[1, i].imshow(fake[i, 0], cmap="magma", vmin=vmin, vmax=vmax)
        axes[1, i].axis("off")

    axes[0, 0].set_ylabel("real", fontsize=12)
    axes[1, 0].set_ylabel("generated", fontsize=12)
    epoch = run_state[run_name].get("loaded_epoch")
    fig.suptitle(f"{RUNS[run_name]['label']} | epoch {epoch} | display [{vmin:.3f}, {vmax:.3f}]", y=1.02)
    fig.tight_layout()
    plt.show()

for run_name in run_state:
    plot_real_fake_grid(run_name, n=min(8, N_GEN, N_REAL))


## Pixel Histograms

This checks the one-point distribution. A larger model can look visually better but still fail this distribution test.


In [ ]:
def summarize_tensor(name, x):
    x = x.detach().cpu().flatten()
    qs = torch.quantile(x, torch.tensor([0.0, 0.001, 0.01, 0.1, 0.5, 0.9, 0.99, 0.999, 1.0]))
    print(name)
    print(f"  min/median/max = {qs[0].item():.4f} / {qs[4].item():.4f} / {qs[-1].item():.4f}")
    print(f"  q99/q99.9      = {qs[6].item():.4f} / {qs[7].item():.4f}")
    print(f"  mean/std       = {x.mean().item():.6f} / {x.std().item():.6f}")
    print(f"  frac(|x|>=.999) = {(x.abs() >= 0.999).float().mean().item():.8e}")

active_runs = list(run_state)
fig, axes = plt.subplots(1, len(active_runs), figsize=(6 * len(active_runs), 4), sharey=True)
axes = np.atleast_1d(axes)
bins = np.linspace(-1, 1, 120)

for ax, run_name in zip(axes, active_runs):
    real = real_images[run_name].numpy().ravel()
    fake = generated[run_name].numpy().ravel()
    ax.hist(real, bins=bins, density=True, histtype="step", lw=2, color="black", label="real")
    ax.hist(fake, bins=bins, density=True, histtype="step", lw=2, color=RUNS[run_name]["color"], label="generated")
    ax.set_yscale("log")
    ax.set_title(RUNS[run_name]["label"])
    ax.set_xlabel("normalized pixel value")
    ax.legend()
axes[0].set_ylabel("density")
fig.tight_layout()
plt.show()

for run_name in active_runs:
    print("\n" + run_name)
    summarize_tensor("real", real_images[run_name])
    summarize_tensor("generated", generated[run_name])


## Radial 2D Power Spectrum

The main diagnostic is `generated P(k) / real P(k)`, computed in the same transformed space for each run.


In [ ]:
def radial_power_spectrum_2d(field, nbins=25):
    field = np.asarray(field, dtype=np.float64)
    field = field - field.mean()

    fft = np.fft.fftn(field)
    power = (fft * fft.conj()).real / field.size

    ky = np.fft.fftfreq(field.shape[0]) * field.shape[0]
    kx = np.fft.fftfreq(field.shape[1]) * field.shape[1]
    kkx, kky = np.meshgrid(kx, ky)
    kvals = np.sqrt(kkx**2 + kky**2)

    valid = kvals > 0
    edges = np.linspace(kvals[valid].min(), kvals[valid].max(), nbins + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])

    pk = np.full(nbins, np.nan)
    for i in range(nbins):
        mask = (kvals >= edges[i]) & (kvals < edges[i + 1])
        if mask.any():
            pk[i] = power[mask].mean()
    return pk, centers


def batch_power_spectra(images, nbins=25):
    arr = images.detach().cpu().numpy()
    pks = []
    kbins = None
    for img in arr:
        pk, kbins = radial_power_spectrum_2d(img[0], nbins=nbins)
        pks.append(pk)
    return np.asarray(pks), kbins

pk = {}
for run_name in run_state:
    real_pk, kbins = batch_power_spectra(real_images[run_name], NBINS)
    fake_pk, _ = batch_power_spectra(generated[run_name], NBINS)
    pk[run_name] = {"real": real_pk, "fake": fake_pk, "kbins": kbins}

print("computed P(k) for:", list(pk.keys()))


In [ ]:
def band_summary(ratio):
    finite = np.where(np.isfinite(ratio))[0]
    if len(finite) == 0:
        return np.nan, np.nan, np.nan
    thirds = np.array_split(finite, 3)
    return tuple(float(np.nanmean(ratio[t])) for t in thirds)

print("Mean P(k) ratio relative to each run's real-data mean:")
summary_rows = []
for run_name in pk:
    real_mean = np.nanmean(pk[run_name]["real"], axis=0)
    fake_mean = np.nanmean(pk[run_name]["fake"], axis=0)
    ratio = fake_mean / np.clip(real_mean, 1e-30, None)
    low, mid, high = band_summary(ratio)
    row = {"run": run_name, "low_k": low, "mid_k": mid, "high_k": high}
    summary_rows.append(row)
    print(f"  {run_name:6s} | low-k={low:.3f}  mid-k={mid:.3f}  high-k={high:.3f}")

try:
    import pandas as pd
    display(pd.DataFrame(summary_rows))
except Exception:
    pass


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for run_name in pk:
    color = RUNS[run_name]["color"]
    kbins = pk[run_name]["kbins"]
    real_mean = np.nanmean(pk[run_name]["real"], axis=0)
    real_std = np.nanstd(pk[run_name]["real"], axis=0)
    fake_mean = np.nanmean(pk[run_name]["fake"], axis=0)

    axes[0].plot(kbins, real_mean, color=color, lw=2, ls=":", label=f"{run_name} real")
    axes[0].plot(kbins, fake_mean, color=color, lw=2, label=f"{run_name} generated")
    axes[0].fill_between(
        kbins,
        np.clip(real_mean - real_std, 1e-30, None),
        real_mean + real_std,
        color=color,
        alpha=0.12,
    )

axes[0].set_yscale("log")
axes[0].set_xlabel("k bin")
axes[0].set_ylabel("mean P(k)")
axes[0].set_title("Transformed-space P(k)")
axes[0].legend(fontsize=9)

for run_name in pk:
    color = RUNS[run_name]["color"]
    kbins = pk[run_name]["kbins"]
    real_mean = np.nanmean(pk[run_name]["real"], axis=0)
    fake_mean = np.nanmean(pk[run_name]["fake"], axis=0)
    ratio = fake_mean / np.clip(real_mean, 1e-30, None)
    axes[1].plot(kbins, ratio, color=color, lw=2.5, label=f"{run_name} generated / real")

axes[1].axhline(1.0, color="black", lw=1.5, ls=":")
axes[1].set_xlabel("k bin")
axes[1].set_ylabel("mean generated P(k) / real mean P(k)")
axes[1].set_title("Main diagnostic")
axes[1].legend()
fig.tight_layout()
plt.show()


## PCA Encoder Diagnostics

The PCA encoder is a lightweight feature extractor. It is fitted on real images only:

```text
image -> flatten -> subtract real-data mean image -> project onto top PCA components
```

Then real and generated images are compared in this low-dimensional PCA feature space. This is useful because it checks large-scale image structure beyond just pixel mean/std, but it is still not a replacement for `P(k)`.


In [ ]:
PCA_RANK = 16
PCA_TRAIN_RUN = "run16" if "run16" in real_images else next(iter(real_images))

pca_train = real_images[PCA_TRAIN_RUN].double()
rank = min(PCA_RANK, pca_train.shape[0] - 1)
print("PCA training run:", PCA_TRAIN_RUN)
print("PCA rank:", rank)

pca_encoder = build_pca_encoder(pca_train, rank=rank).double()

# Compute explained variance for interpretation. The repo encoder stores V and center,
# but not the singular values, so we recompute them here for diagnostics.
X = pca_train.flatten(1)
X_centered = X - X.mean(0)
_, S, _ = torch.linalg.svd(X_centered, full_matrices=False)
explained = (S**2) / torch.clamp((S**2).sum(), min=1e-30)
explained_rank = explained[:rank]
print("Explained variance in first", rank, "PCs:", float(explained_rank.sum()))

plt.figure(figsize=(6, 4))
plt.plot(np.arange(1, rank + 1), explained_rank.cpu().numpy(), marker="o")
plt.xlabel("PCA component")
plt.ylabel("explained variance fraction")
plt.title(f"PCA spectrum fitted on {PCA_TRAIN_RUN} real images")
plt.tight_layout()
plt.show()


In [ ]:
pca_features = {}
for run_name in run_state:
    with torch.no_grad():
        feats_real = pca_encoder(real_images[run_name].double()).cpu()
        feats_fake = pca_encoder(generated[run_name].double()).cpu()
    pca_features[run_name] = {"real": feats_real, "fake": feats_fake}
    print(run_name, "real features", tuple(feats_real.shape), "fake features", tuple(feats_fake.shape))


In [ ]:
fig, axes = plt.subplots(1, len(pca_features), figsize=(5 * len(pca_features), 4), sharex=True, sharey=True)
axes = np.atleast_1d(axes)

for ax, run_name in zip(axes, pca_features):
    feats_real = pca_features[run_name]["real"]
    feats_fake = pca_features[run_name]["fake"]
    ax.scatter(feats_real[:, 0], feats_real[:, 1], s=18, c="black", alpha=0.45, label="real")
    ax.scatter(feats_fake[:, 0], feats_fake[:, 1], s=28, c=RUNS[run_name]["color"], alpha=0.75, label="generated")
    ax.set_title(RUNS[run_name]["label"])
    ax.set_xlabel("PC1")
    ax.legend()
axes[0].set_ylabel("PC2")
fig.suptitle("Real vs generated in PCA feature space", y=1.04)
fig.tight_layout()
plt.show()


## PCA Metric Definitions With Equations

Let each image be flattened into a vector:

$$
 x_i \in \mathbb{R}^D,
$$

where $D = C \times H \times W$. In this notebook, $C=1$ and $H=W=128$.

The PCA basis is fitted using real images from `run16`. Let the real training matrix be:

$$
X_{\rm real} = \{x_1, x_2, \ldots, x_N\}.
$$

First compute the real-data mean image:

$$
\mu_x = \frac{1}{N}\sum_{i=1}^{N} x_i.
$$

Then PCA finds the top `rank` orthonormal directions:

$$
V = [v_1, v_2, \ldots, v_r] \in \mathbb{R}^{D \times r}.
$$

The PCA feature vector for any image $x$ is:

$$
z = (x - \mu_x)^T V.
$$

So each image becomes:

$$
x \in \mathbb{R}^{D} \quad \rightarrow \quad z \in \mathbb{R}^{r}.
$$

For one run, define real PCA features and generated PCA features:

$$
Z_r = \{z^r_1, \ldots, z^r_N\}, \qquad
Z_g = \{z^g_1, \ldots, z^g_M\}.
$$

### `pca_fid`

This is the Fréchet distance between Gaussian approximations to the real and generated PCA feature distributions.

Compute feature means:

$$
\mu_r = \frac{1}{N}\sum_{i=1}^{N} z^r_i,
\qquad
\mu_g = \frac{1}{M}\sum_{i=1}^{M} z^g_i.
$$

Compute feature covariances:

$$
\Sigma_r = \frac{1}{N-1}\sum_{i=1}^{N}(z^r_i - \mu_r)(z^r_i - \mu_r)^T,
$$

$$
\Sigma_g = \frac{1}{M-1}\sum_{i=1}^{M}(z^g_i - \mu_g)(z^g_i - \mu_g)^T.
$$

Then:

$$
\mathrm{PCA\text{-}FID}
= \|\mu_r - \mu_g\|_2^2
+ \mathrm{Tr}(\Sigma_r)
+ \mathrm{Tr}(\Sigma_g)
- 2\,\mathrm{Tr}\left((\Sigma_r^{1/2}\Sigma_g\Sigma_r^{1/2})^{1/2}\right).
$$

Lower is better. A perfect match would be $0$.

### `pca_kid_mean`

This is a Kernel MMD estimate between real and generated PCA features using a polynomial kernel.

The polynomial kernel is:

$$
k(a,b) = (\gamma a^T b + c)^d,
$$

where in the notebook:

$$
d = 3, \qquad c = 1, \qquad \gamma = \frac{1}{r}.
$$

For subsets of size $n$, the unbiased squared MMD estimate is:

$$
\mathrm{MMD}^2
= \frac{1}{n(n-1)}\sum_{i\neq j} k(z^r_i,z^r_j)
+ \frac{1}{n(n-1)}\sum_{i\neq j} k(z^g_i,z^g_j)
- \frac{2}{n^2}\sum_{i,j} k(z^r_i,z^g_j).
$$

The notebook repeats this over random subsets. Then:

$$
\mathrm{pca\_kid\_mean} = \mathrm{mean}(\mathrm{MMD}^2 \text{ over subsets}).
$$

Lower is better. Because this is an unbiased finite-sample estimator, it can be slightly negative when sample size is small.

### `pca_kid_std`

This is the standard deviation of the KID estimates over random subsets:

$$
\mathrm{pca\_kid\_std}
= \mathrm{std}(\mathrm{MMD}^2 \text{ over subsets}).
$$

Smaller means the KID estimate is more stable.

### `pca_mean_l2`

This measures how far the generated PCA-feature mean is from the real PCA-feature mean:

$$
\mathrm{pca\_mean\_l2}
= \|\mu_g - \mu_r\|_2.
$$

Lower is better. Large value means generated samples are shifted to the wrong location in PCA space.

### `pca_std_l2`

Let the component-wise PCA standard deviations be:

$$
s_r = \mathrm{std}(Z_r) \in \mathbb{R}^{r},
\qquad
s_g = \mathrm{std}(Z_g) \in \mathbb{R}^{r}.
$$

Then:

$$
\mathrm{pca\_std\_l2}
= \|s_g - s_r\|_2.
$$

Lower is better. Large value means generated samples have the wrong amount of variation across PCA modes.

### PCA Mean Difference Plot

For PCA component $j$:

$$
\Delta\mu_j = \mu_{g,j} - \mu_{r,j}.
$$

The left plot shows $\Delta\mu_j$ for each component. Perfect match is the horizontal zero line.

### PCA Std Ratio Plot

For PCA component $j$:

$$
R_j = \frac{s_{g,j}}{s_{r,j}}.
$$

The right plot shows $R_j$ for each component. Perfect match is the horizontal one line.

Interpretation:

- $R_j > 1$: generated samples vary too much along PCA component $j$.
- $R_j < 1$: generated samples are too collapsed along PCA component $j$.
- $R_j = 1$: generated and real samples have matching spread along PCA component $j$.


In [ ]:
pca_metric_rows = []
for run_name, feats in pca_features.items():
    feats_real = feats["real"].double()
    feats_fake = feats["fake"].double()
    subset_size = min(len(feats_real), len(feats_fake), 1000)

    fid = compute_fid(feats_real, feats_fake)
    if subset_size >= 2:
        kid_mean, kid_std = compute_kid(
            feats_real,
            feats_fake,
            subset_size=subset_size,
            n_subsets=20,
        )
    else:
        kid_mean, kid_std = float("nan"), float("nan")

    mean_l2 = torch.linalg.vector_norm(feats_fake.mean(0) - feats_real.mean(0)).item()
    std_l2 = torch.linalg.vector_norm(feats_fake.std(0) - feats_real.std(0)).item()

    pca_metric_rows.append({
        "run": run_name,
        "epoch": run_state[run_name].get("loaded_epoch"),
        "pca_fid": fid,
        "pca_kid_mean": kid_mean,
        "pca_kid_std": kid_std,
        "pca_mean_l2": mean_l2,
        "pca_std_l2": std_l2,
    })

pca_metric_rows = sorted(pca_metric_rows, key=lambda r: r["pca_fid"])
try:
    import pandas as pd
    display(pd.DataFrame(pca_metric_rows))
except Exception:
    for row in pca_metric_rows:
        print(json.dumps(row, indent=2))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
components = np.arange(1, rank + 1)

for run_name, feats in pca_features.items():
    color = RUNS[run_name]["color"]
    real = feats["real"]
    fake = feats["fake"]
    mean_diff = (fake.mean(0) - real.mean(0)).numpy()
    std_ratio = (fake.std(0) / torch.clamp(real.std(0), min=1e-12)).numpy()
    axes[0].plot(components, mean_diff, marker="o", color=color, label=run_name)
    axes[1].plot(components, std_ratio, marker="o", color=color, label=run_name)

axes[0].axhline(0.0, color="black", ls=":", lw=1.5)
axes[0].set_title("Generated - real PCA mean")
axes[0].set_xlabel("PCA component")
axes[0].set_ylabel("feature mean difference")

axes[1].axhline(1.0, color="black", ls=":", lw=1.5)
axes[1].set_title("Generated / real PCA std")
axes[1].set_xlabel("PCA component")
axes[1].set_ylabel("feature std ratio")
axes[1].legend()
fig.tight_layout()
plt.show()


## Quick Ranking

This rough score combines `P(k)` log-error with pixel mean/std mismatch. Smaller is better. Use it only as a guide; still inspect the plots.


In [ ]:
scores = []
for run_name in pk:
    real_mean = np.nanmean(pk[run_name]["real"], axis=0)
    fake_mean = np.nanmean(pk[run_name]["fake"], axis=0)
    ratio = fake_mean / np.clip(real_mean, 1e-30, None)
    pk_log_mae = float(np.nanmean(np.abs(np.log10(np.clip(ratio, 1e-12, None)))))

    real_flat = real_images[run_name].flatten().float()
    fake_flat = generated[run_name].flatten().float()
    mean_err = float(abs(fake_flat.mean() - real_flat.mean()))
    std_err = float(abs(fake_flat.std() - real_flat.std()))

    scores.append({
        "run": run_name,
        "epoch": run_state[run_name].get("loaded_epoch"),
        "pk_log10_mae": pk_log_mae,
        "mean_abs_err": mean_err,
        "std_abs_err": std_err,
        "combined": pk_log_mae + mean_err + std_err,
    })

scores = sorted(scores, key=lambda r: r["combined"])
try:
    import pandas as pd
    display(pd.DataFrame(scores))
except Exception:
    for row in scores:
        print(json.dumps(row, indent=2))


## Interpretation

Compare carefully because the epoch budgets differ.

- `run9` answers whether StepLR helped at the same U64 size.
- `run10` answers whether the U128 model improves the field before going all the way to U256.
- `run11` answers whether the U256-width model improves results enough to justify the cost.
- If `run10` beats `run16` but `run11` does not, U256 may be too expensive or harder to optimize.
- If `run11` is still early, do not conclude much from sample quality yet; first check whether loss is still falling.
